In [27]:
from functools import reduce
import itertools
from collections import defaultdict

import yfinance as yf
import polars as pl
import polars.selectors as cs

import infomeasure as im
import dcor

In [75]:
names = {
    #'VIXCLS':         'VIX',        # CBOE VIX
    #'BAMLHE00EHY0EY': 'EU_HY_OAS',  # ICE BofA Euro HY OAS TODO
    #'ECBDFR':         'ECB_RATE',   # ECB deposit facility rate
    #'T10Y2Y':         'US_T10Y2Y',   # US 10Y–2Y yield spread
    '^GSPC': 'S&P 500', 
    '^VIX': 'VIX', 
    #'^V2TX' : 'V2TX',
    'DX-Y.NYB': 'Dollar Index', 
    'WTI': 'Crude Oil', 
    #'^TNX': '10-Year Treasury', 
    #'^IRX': '3-Month Treasury'
}
data = pl.from_pandas(
    yf.download(list(names.keys()), period='10y', interval='1d', multi_level_index=False).Close.reset_index(),
    schema_overrides={'Date': pl.Date}
    )
uncertainty =  pl.read_excel('https://policyuncertainty.com/media/Europe_Policy_Uncertainty_Data.xlsx', infer_schema_length=10_000)\
.select(pl.date('Year', 'Month', 1).alias('Date'), cs.numeric().exclude('Month'))
    #.filter(pl.concat_list(pl.exclude('Year', 'Month')).list.drop_nulls().list.len() > 4)\

v2x = pl.read_csv('https://www.stoxx.com/document/Indices/Current/HistoricalData/h_v2tx.txt', separator=';')\
    .select(pl.col.Date.str.to_date(), pl.col.Indexvalue.alias('V2TX'))
    #.group_by(Date=pl.col.Date.str.to_date().dt.strftime('%Y-%m-01').str.to_date(), maintain_order=True).agg(V2TX=pl.col.Indexvalue.mean())

brent = pl.read_csv('https://datahub.io/core/oil-prices/_r/-/data/brent-daily.csv', try_parse_dates=True)\
    .rename(dict(Price='Brent'))
    #.group_by(Date=pl.col.Date.dt.strftime('%Y-%m-01').str.to_date(), maintain_order=True).agg(pl.col.Price.mean())

#hy_oas = pl.scan_csv(
#    'https://fred.stlouisfed.org/graph/fredgraph.csv?bgcolor=%23ebf3fb&chart_type=line&drp=0&fo=open%20sans&graph_bgcolor=%23ffffff&height=450&mode=fred&recession_bars=off&txtcolor=%23444444&ts=12&tts=12&width=1140&nt=0&thu=0&trc=0&show_legend=yes&show_axis_titles=yes&show_tooltip=yes&id=BAMLHE00EHYIEY&scale=left&cosd=2023-06-05&coed=2026-06-02&line_color=%230073e6&link_values=false&line_style=solid&mark_type=none&mw=3&lw=3&ost=-99999&oet=99999&mma=0&fml=a&fq=Daily%2C%20Close&fam=avg&fgst=lin&fgsnd=2010-06-05&line_index=1&transformation=lin'
#)
#ig_oas = pl.scan_csv('
#    'https://fred.stlouisfed.org/graph/fredgraph.csv?bgcolor=%23ebf3fb&chart_type=line&drp=0&fo=open%20sans&graph_bgcolor=%23ffffff&height=450&mode=fred&recession_bars=off&txtcolor=%23444444&ts=12&tts=12&width=1140&nt=0&thu=0&trc=0&show_legend=yes&show_axis_titles=yes&show_tooltip=yes&id=BAMLHE4HEY4&scale=left&cosd=2023-06-05&coed=2026-06-02&line_color=%230073e6&link_values=false&line_style=solid&mark_type=none&mw=3&lw=3&ost=-99999&oet=99999&mma=0&fml=a&fq=Daily%2C%20Close&fam=avg&fgst=lin&fgsnd=2010-06-05&line_index=1&transformation=lin'
#)



df = reduce(lambda x, y: x.join(y, on='Date', how='left'), [
    data,
    uncertainty,
    v2x,
    brent
]).with_columns(pl.exclude('Date').forward_fill()).drop_nulls()
df.null_count()

[*********************100%***********************]  4 of 4 completed


Date,DX-Y.NYB,WTI,^GSPC,^VIX,European_News_Index,Germany_News_Index,Italy_News_Index,UK_News_Index,France_News_Index,Spain_News_Index,V2TX,Brent
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0


In [76]:
import sys
sys.path.insert(0, '..')
from src.signatures import rolling, log, signature

In [77]:
sigs = signature(*map(lambda x: x.forward_fill().pct_change(), map(pl.col, df.columns[1:])), level=1)
rolling_feats = {f"{','.join(k)}_{w}M" : v for w in [3, 6, 12] for k, v in log(rolling(sigs, window_size=w)).items()}
df.select(**rolling_feats).null_count()

DX-Y.NYB_3M,WTI_3M,^GSPC_3M,^VIX_3M,European_News_Index_3M,Germany_News_Index_3M,Italy_News_Index_3M,UK_News_Index_3M,France_News_Index_3M,Spain_News_Index_3M,V2TX_3M,Brent_3M,DX-Y.NYB_6M,WTI_6M,^GSPC_6M,^VIX_6M,European_News_Index_6M,Germany_News_Index_6M,Italy_News_Index_6M,UK_News_Index_6M,France_News_Index_6M,Spain_News_Index_6M,V2TX_6M,Brent_6M,DX-Y.NYB_12M,WTI_12M,^GSPC_12M,^VIX_12M,European_News_Index_12M,Germany_News_Index_12M,Italy_News_Index_12M,UK_News_Index_12M,France_News_Index_12M,Spain_News_Index_12M,V2TX_12M,Brent_12M
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [78]:
macro_sigs = df.select('Date', **rolling_feats).drop_nulls()
macro_sigs = macro_sigs.select('Date', *itertools.compress(macro_sigs.columns[1:], (macro_sigs.std() > 0.1).row()[1:]))
macro_sigs

Date,^VIX_3M,Italy_News_Index_3M,UK_News_Index_3M,Spain_News_Index_3M,V2TX_3M,WTI_6M,^VIX_6M,Germany_News_Index_6M,Italy_News_Index_6M,UK_News_Index_6M,France_News_Index_6M,Spain_News_Index_6M,V2TX_6M,WTI_12M,^VIX_12M,European_News_Index_12M,Germany_News_Index_12M,Italy_News_Index_12M,UK_News_Index_12M,France_News_Index_12M,Spain_News_Index_12M,V2TX_12M
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2016-07-05,0.054841,0.0,0.0,0.0,0.067383,-0.04386,0.054841,0.0,0.0,0.0,0.0,0.0,0.067383,-0.04386,0.054841,0.0,0.0,0.0,0.0,0.0,0.0,0.067383
2016-07-06,0.015046,0.0,0.0,0.0,0.125514,-0.002575,0.015046,0.0,0.0,0.0,0.0,0.0,0.125514,-0.002575,0.015046,0.0,0.0,0.0,0.0,0.0,0.0,0.125514
2016-07-07,-0.053164,0.0,0.0,0.0,-0.002486,-0.046628,0.001677,0.0,0.0,0.0,0.0,0.0,0.064897,-0.046628,0.001677,0.0,0.0,0.0,0.0,0.0,0.0,0.064897
2016-07-08,-0.11906,0.0,0.0,0.0,-0.13332,-0.04202,-0.104014,0.0,0.0,0.0,0.0,0.0,-0.007806,-0.04202,-0.104014,0.0,0.0,0.0,0.0,0.0,0.0,-0.007806
2016-07-11,-0.079933,0.0,0.0,0.0,-0.115144,-0.046607,-0.078256,0.0,0.0,0.0,0.0,0.0,-0.050248,-0.046607,-0.078256,0.0,0.0,0.0,0.0,0.0,0.0,-0.050248
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-05-29,-0.060447,0.0,0.0,0.0,-0.02178,-0.177773,-0.084045,0.0,0.0,0.0,0.0,0.0,-0.087783,-0.174832,-0.112789,0.0,0.0,0.0,0.0,0.0,0.0,-0.08614
2026-06-01,0.020966,0.0,0.0,0.0,0.032979,-0.093534,-0.029808,0.0,0.0,0.0,0.0,0.0,0.01541,-0.158009,-0.132926,0.0,0.0,0.0,0.0,0.0,0.0,-0.096616
2026-06-02,0.030205,0.0,0.0,0.0,-0.003011,-0.004751,-0.07257,0.0,0.0,0.0,0.0,0.0,-0.035097,-0.184174,-0.117273,0.0,0.0,0.0,0.0,0.0,0.0,-0.184665


In [82]:
etf_tickers = [
    'MCEU.PA',
    'QCEU.PA',
    'MIVO.PA',
    'CV9.PA',
    'VAL.PA',
    #'LCEU.PA',
    #'DGEU.PA',
    #'VCEU.PA',
    #'GCEU.PA',
]
etfs = pl.from_pandas(yf.download(etf_tickers, start='2022-01-01').Close.reset_index(), schema_overrides={'Date': pl.Date}).drop_nulls()
#esg_etfs = esg_etfs.group_by(pl.col.Date.dt.strftime('%Y-%m-01').str.to_date(), maintain_order=True).agg(pl.exclude('Date').mean())
etfs

[*********************100%***********************]  5 of 5 completed


Date,CV9.PA,MCEU.PA,MIVO.PA,QCEU.PA,VAL.PA
date,f64,f64,f64,f64,f64
2024-06-17,299.423096,105.239998,120.440002,113.639999,118.878983
2024-06-18,301.708191,106.18,120.440002,114.18,119.919006
2024-06-19,302.205902,106.080002,120.440002,114.0,119.826141
2024-06-20,304.653198,107.32,120.440002,114.800003,120.921898
2024-06-21,302.565186,106.080002,120.440002,114.120003,119.789009
…,…,…,…,…,…
2026-05-28,442.880096,148.440002,120.440002,120.959999,181.300003
2026-05-29,442.450714,148.179993,120.440002,120.82,181.759995
2026-06-01,440.559814,147.740005,120.440002,119.120003,180.860001


In [93]:

features = defaultdict(list)
shifted_esg_etfs = etfs.select('Date', pl.exclude('Date').diff(-255)).drop_nulls()
X, y = pl.align_frames(macro_sigs, shifted_esg_etfs, on='Date', how='inner')
X, y = X.drop('Date'), y.drop('Date')
print(X.shape, y.shape)
for macro, esg in itertools.product(X, y):
    mi = im.mutual_information(esg, macro, approach='kernel', kernel='box', bandwidth=0.5)
    distance_corr = dcor.distance_correlation(esg, macro)
    if mi > 0.2 or distance_corr > 0.2:
            print(f"ESG: {esg.name}, Macro: {macro.name}, MI: {mi:.4f}, Distance Corr: {distance_corr:.4f}")
            features[esg.name].append(macro.name)

features

(238, 22) (238, 5)
ESG: MCEU.PA, Macro: ^VIX_3M, MI: 0.1126, Distance Corr: 0.2222
ESG: CV9.PA, Macro: WTI_6M, MI: 0.0352, Distance Corr: 0.2028
ESG: MCEU.PA, Macro: WTI_6M, MI: 0.0047, Distance Corr: 0.2034
ESG: VAL.PA, Macro: WTI_6M, MI: 0.0303, Distance Corr: 0.2005
ESG: CV9.PA, Macro: ^VIX_6M, MI: 0.3605, Distance Corr: 0.1941
ESG: MCEU.PA, Macro: ^VIX_6M, MI: 0.2254, Distance Corr: 0.2878
ESG: QCEU.PA, Macro: ^VIX_6M, MI: 0.2138, Distance Corr: 0.1453
ESG: VAL.PA, Macro: ^VIX_6M, MI: 0.2417, Distance Corr: 0.2149
ESG: CV9.PA, Macro: Germany_News_Index_6M, MI: 0.2458, Distance Corr: 0.1373
ESG: CV9.PA, Macro: Italy_News_Index_6M, MI: 0.2296, Distance Corr: 0.1456
ESG: CV9.PA, Macro: UK_News_Index_6M, MI: 0.2729, Distance Corr: 0.1433
ESG: MCEU.PA, Macro: UK_News_Index_6M, MI: 0.1861, Distance Corr: 0.2008
ESG: CV9.PA, Macro: France_News_Index_6M, MI: 0.2621, Distance Corr: 0.1607
ESG: CV9.PA, Macro: V2TX_6M, MI: 0.2609, Distance Corr: 0.1910
ESG: MCEU.PA, Macro: V2TX_6M, MI: 0.1992

defaultdict(list,
            {'MCEU.PA': ['^VIX_3M',
              'WTI_6M',
              '^VIX_6M',
              'UK_News_Index_6M',
              'V2TX_6M',
              'WTI_12M',
              '^VIX_12M',
              'European_News_Index_12M',
              'Germany_News_Index_12M',
              'Italy_News_Index_12M',
              'UK_News_Index_12M',
              'France_News_Index_12M',
              'Spain_News_Index_12M',
              'V2TX_12M'],
             'CV9.PA': ['WTI_6M',
              '^VIX_6M',
              'Germany_News_Index_6M',
              'Italy_News_Index_6M',
              'UK_News_Index_6M',
              'France_News_Index_6M',
              'V2TX_6M',
              'WTI_12M',
              '^VIX_12M',
              'European_News_Index_12M',
              'Germany_News_Index_12M',
              'Italy_News_Index_12M',
              'UK_News_Index_12M',
              'France_News_Index_12M',
              'Spain_News_Index_12M',
              '

In [94]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
for target, feats in features.items():
    _X = X.select(pl.col(feats)).to_numpy()
    _y = y.get_column(target).to_numpy().ravel()
    _X_train, _X_test, _y_train, _y_test = train_test_split(_X, _y, test_size=0.3)
    #model = RandomForestRegressor(n_estimators=10, random_state=42)
    model = DecisionTreeRegressor(criterion='squared_error')
    model.fit(_X_train, _y_train)
    print(f"Feature importances for predicting {target}:")
    for feat, importance in sorted(zip(feats, model.feature_importances_), key=lambda a: a[1], reverse=True)[:10]:
        print(f"{feat}: {importance:.4f}")
    print('\n')
    print(f"Model score for predicting {target}: {model.score(_X_test, _y_test):.4f}")
    print('\n\n')

Feature importances for predicting MCEU.PA:
V2TX_12M: 0.3408
WTI_12M: 0.1619
WTI_6M: 0.1233
^VIX_12M: 0.0986
France_News_Index_12M: 0.0807
V2TX_6M: 0.0622
^VIX_6M: 0.0561
^VIX_3M: 0.0399
Italy_News_Index_12M: 0.0346
UK_News_Index_6M: 0.0019


Model score for predicting MCEU.PA: 0.2731



Feature importances for predicting CV9.PA:
Italy_News_Index_12M: 0.2021
WTI_12M: 0.1697
V2TX_12M: 0.1585
V2TX_6M: 0.1268
European_News_Index_12M: 0.1039
^VIX_12M: 0.1022
WTI_6M: 0.0962
^VIX_6M: 0.0300
UK_News_Index_12M: 0.0087
Germany_News_Index_12M: 0.0011


Model score for predicting CV9.PA: 0.0604



Feature importances for predicting VAL.PA:
^VIX_12M: 0.3591
WTI_12M: 0.2212
V2TX_12M: 0.1917
^VIX_6M: 0.0932
WTI_6M: 0.0520
Germany_News_Index_12M: 0.0518
France_News_Index_12M: 0.0159
Italy_News_Index_12M: 0.0143
European_News_Index_12M: 0.0008
UK_News_Index_12M: 0.0000


Model score for predicting VAL.PA: 0.1713



Feature importances for predicting QCEU.PA:
WTI_12M: 0.2398
European_News_Index_12M: 0.

In [95]:
y

CV9.PA,MCEU.PA,MIVO.PA,QCEU.PA,VAL.PA
f64,f64,f64,f64,f64
-57.565704,-18.700005,0.0,2.919998,-28.362473
-54.525818,-17.659996,0.0,4.160004,-27.032303
-50.34671,-16.340004,0.0,5.980003,-25.797279
-51.058716,-16.979996,0.0,5.660004,-26.233803
-51.394379,-17.760002,0.0,5.419998,-26.26841
…,…,…,…,…
-80.091583,-21.900002,0.0,-8.199997,-31.563248
-80.750702,-21.339996,0.0,-7.900002,-31.810486
-80.882904,-21.720009,0.0,-7.080002,-31.819611
